In [1]:
%load_ext autoreload
%autoreload 2

In [7]:
# %reload model

# from model import model_parts as mp
from model import ProtENN2_style

In [ ]:
import numpy as np
import torch

model = ProtENN2_style()

test_input = torch.rand(64, 100, 21)

output = model(test_input)

In [9]:
print(output.shape)

print(torch.sum(output, axis=-1).shape)

torch.Size([64, 100, 100])
torch.Size([64, 100])


In [49]:
import pandas as pd

dataset_df = pd.read_pickle("../../dataset/dataset.pkl")  
dataset_df




,pfam_tensor,sequence
A1AL94,"[None, None, None, None, None, None, None, Non...",MSLTTLVTQQAPDFTAEAVMADNSFASITLSSLKGKFVLLLFYPLD...
A0A0F7V1V9,"[None, None, None, None, None, None, None, Non...",MAACLRAARLSLRQMEGLIEPSVRGRSSPLSMVRLLPSSSVSSSSP...
Q4UGC8,"[None, None, None, None, None, None, None, Non...",MKLTGISLISSLSYIRNTLPLKNTLTAFHTLNTRNNLKSVNRITSV...
A4H879,"[None, None, None, None, None, None, None, Non...",MSCGDAKMNEPAPPFEEMALMPNGSFKKINLASYKGKWVVLFFYPL...
A0A0M5IXP6,"[None, None, None, None, None, None, None, Non...",MNPSANESGVCSPVQIGDAAPNFQARTTLGEMSLSDYRGRWVLLFS...
...,...,...
U4UHL9,"[None, None, None, None, None, None, None, Non...",MANTNTDLTEEDAADLQFPKGITSNKLLCGNRFLNLLDSRPQLPTT...
A0A6J2XE68,"[None, None, None, None, None, None, None, Non...",MANVNTDLTEEDAADLQFPKDSTSQIPTTSTLVNVSDKNVILGYCS...
A0A834MIF3,"[None, None, None, None, None, None, None, Non...",MANVNTDLTEEDAADLQFPKDATSQKPSTSSVVNVSDKSIILGYCS...
A0A653BD21,"[None, None, None, None, None, None, None, Non...",MANTNTDLTEEDAADLQFPKADELAKTTLSYVSTNTFLDNPDSSYC...


In [50]:
# filter out sequences with more than 100 amino acids
dataset_df["seq_len"] = dataset_df["sequence"].apply(lambda x: len(x))
dataset_df = dataset_df[dataset_df["seq_len"] <= 100]

In [51]:
subset_df = dataset_df.head(10000)
subset_df

,pfam_tensor,sequence,seq_len
G2Z1S9,"[None, None, None, None, None, None, None, Non...",MENYNQIDERYIAAQKRVQEIKGFYGHLASYVLVNLFLLILNLVSS...,96
A4CNS3,"[None, None, None, None, None, None, None, Non...",MEDFKKESKYIRARERVEELKKFYGNVASYIFVITLLGIINYLTYW...,100
G0L8V4,"[None, None, None, None, None, None, None, PF1...",MERESKYIRAKERVEREKKFYNGLISYVVTISFLAAINYYTNGFAY...,97
A4CNS4,"[None, None, None, None, None, None, None, Non...",MEDFSHADRYLRAKKRVEDIRGFYGNLITYLIVIPFLIWLNWRTTS...,93
F9YQQ0,"[None, None, None, None, None, None, None, Non...",MNTIEPNAYRKAEKRVKKLKKFYHHLATYVVVNTFLVGLNLYQTPN...,93
...,...,...,...
Q1GQN9,"[None, None, PF09413.15, PF09413.15, PF09413.1...",MPLVELVRLPNGAEAELLRGRLESAGVHAVCFDAGMNIAESVGLMI...,70
F8KZY9,"[None, None, None, PF09413.15, PF09413.15, PF0...",MNNFVCIYRTFDIVQANLIKSHFENEDILCVLKSNDASGILPHLGF...,74
A8F8N0,"[None, PF09413.15, PF09413.15, PF09413.15, PF0...",MWRILLEEVDLPTANILKSLLEESGIEVLIKPGSFDPVIFGQGGLV...,74
Q8A1E5,"[None, None, None, None, None, None, None, PF0...",MKEEDYSKAIEVFSGSPWEAEIIKGLLESNDIRCVIKDGIMGTLAP...,77


In [ ]:
from model_parts import CUSTOM_ALPHABET
from model import MAX_PROTEIN_LENGTH

# one hot encode sequences and pad to max length
def one_hot_encode(sequence, alphabet=CUSTOM_ALPHABET, max_length=MAX_PROTEIN_LENGTH):
    one_hot = np.zeros((max_length, len(alphabet)), dtype=np.float32)
    for i, char in enumerate(sequence):
        # padding is done with tensor [1,0,0,...]
        if i < max_length and char in alphabet:
            one_hot[i, alphabet[char]] = 1.0
    
    return one_hot

# apply one hot encoding to sequences
subset_df["one_hot"] = subset_df["sequence"].apply(one_hot_encode)

subset_df


/tmp/ipykernel_3910205/2064692852.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset_df["one_hot"] = subset_df["sequence"].apply(one_hot_encode)


,pfam_tensor,sequence,seq_len,one_hot
G2Z1S9,"[None, None, None, None, None, None, None, Non...",MENYNQIDERYIAAQKRVQEIKGFYGHLASYVLVNLFLLILNLVSS...,96,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
A4CNS3,"[None, None, None, None, None, None, None, Non...",MEDFKKESKYIRARERVEELKKFYGNVASYIFVITLLGIINYLTYW...,100,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
G0L8V4,"[None, None, None, None, None, None, None, PF1...",MERESKYIRAKERVEREKKFYNGLISYVVTISFLAAINYYTNGFAY...,97,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
A4CNS4,"[None, None, None, None, None, None, None, Non...",MEDFSHADRYLRAKKRVEDIRGFYGNLITYLIVIPFLIWLNWRTTS...,93,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
F9YQQ0,"[None, None, None, None, None, None, None, Non...",MNTIEPNAYRKAEKRVKKLKKFYHHLATYVVVNTFLVGLNLYQTPN...,93,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
...,...,...,...,...
Q1GQN9,"[None, None, PF09413.15, PF09413.15, PF09413.1...",MPLVELVRLPNGAEAELLRGRLESAGVHAVCFDAGMNIAESVGLMI...,70,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
F8KZY9,"[None, None, None, PF09413.15, PF09413.15, PF0...",MNNFVCIYRTFDIVQANLIKSHFENEDILCVLKSNDASGILPHLGF...,74,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
A8F8N0,"[None, PF09413.15, PF09413.15, PF09413.15, PF0...",MWRILLEEVDLPTANILKSLLEESGIEVLIKPGSFDPVIFGQGGLV...,74,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
Q8A1E5,"[None, None, None, None, None, None, None, PF0...",MKEEDYSKAIEVFSGSPWEAEIIKGLLESNDIRCVIKDGIMGTLAP...,77,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


In [55]:
# get unique pfam_ids
unique_pfams = set()

for pfam_list in subset_df["pfam_tensor"]:
    for pfam in pfam_list:
        unique_pfams.add(pfam)

# create a mapping from pfam_id to index
pfam_to_index = {pfam: i for i, pfam in enumerate(unique_pfams)}

# one hot encode pfam tensors and pad to max length
def pad_pfam_tensor(pfam_tensor, max_length=MAX_PROTEIN_LENGTH):
    padded_tensor = np.zeros((max_length,len(pfam_to_index)), dtype=np.float32)
    for i, pfam in enumerate(pfam_tensor):
        if i < max_length:
            padded_tensor[i, pfam_to_index.get(pfam, 0)] = 1.0  # use 0 for unknown pfams
    return padded_tensor

# apply one hot encoding to pfam tensors
subset_df["pfam_one_hot"] = subset_df["pfam_tensor"].apply(pad_pfam_tensor)

/tmp/ipykernel_3910205/3296820493.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset_df["pfam_one_hot"] = subset_df["pfam_tensor"].apply(pad_pfam_tensor)


In [47]:
pfam_to_index

{'PF02285.20': 0,
 'PF00200.28': 1,
 'PF14138.11': 2,
 'PF12173.13': 3,
 'PF18593.6': 4,
 'PF06620.16': 5,
 'PF13239.11': 6,
 'PF21862.1': 7,
 'PF08132.16': 8,
 'PF02823.21': 9,
 'PF13991.11': 10,
 'PF12095.13': 11,
 'PF00895.25': 12,
 'PF07864.16': 13,
 'PF13356.12': 14,
 'PF05961.16': 15,
 'PF01623.22': 16,
 'PF21627.2': 17,
 'PF06523.16': 18,
 'PF24769.1': 19,
 'PF03477.21': 20,
 'PF13318.11': 21,
 'PF07351.18': 22,
 'PF22629.2': 23,
 'PF22996.2': 24,
 'PF00187.25': 25,
 'PF06747.18': 26,
 'PF06900.16': 27,
 'PF07119.17': 28,
 'PF08590.15': 29,
 'PF14657.12': 30,
 'PF14960.11': 31,
 'PF10515.14': 32,
 'PF08077.16': 33,
 'PF08742.16': 34,
 'PF02038.21': 35,
 'PF06515.16': 36,
 'PF07438.16': 37,
 'PF20077.4': 38,
 'PF20068.4': 39,
 'PF04972.22': 40,
 'PF06803.17': 41,
 'PF06269.17': 42,
 'PF06713.17': 43,
 'PF21801.3': 44,
 'PF17334.7': 45,
 'PF09350.15': 46,
 'PF07190.16': 47,
 'PF07877.16': 48,
 'PF02594.21': 49,
 'PF07256.17': 50,
 'PF07408.16': 51,
 'PF02085.21': 52,
 'PF07484.17'

In [61]:
subset_df

,pfam_tensor,sequence,seq_len,one_hot,pfam_one_hot
G2Z1S9,"[None, None, None, None, None, None, None, Non...",MENYNQIDERYIAAQKRVQEIKGFYGHLASYVLVNLFLLILNLVSS...,96,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
A4CNS3,"[None, None, None, None, None, None, None, Non...",MEDFKKESKYIRARERVEELKKFYGNVASYIFVITLLGIINYLTYW...,100,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
G0L8V4,"[None, None, None, None, None, None, None, PF1...",MERESKYIRAKERVEREKKFYNGLISYVVTISFLAAINYYTNGFAY...,97,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
A4CNS4,"[None, None, None, None, None, None, None, Non...",MEDFSHADRYLRAKKRVEDIRGFYGNLITYLIVIPFLIWLNWRTTS...,93,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
F9YQQ0,"[None, None, None, None, None, None, None, Non...",MNTIEPNAYRKAEKRVKKLKKFYHHLATYVVVNTFLVGLNLYQTPN...,93,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
...,...,...,...,...,...
Q1GQN9,"[None, None, PF09413.15, PF09413.15, PF09413.1...",MPLVELVRLPNGAEAELLRGRLESAGVHAVCFDAGMNIAESVGLMI...,70,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
F8KZY9,"[None, None, None, PF09413.15, PF09413.15, PF0...",MNNFVCIYRTFDIVQANLIKSHFENEDILCVLKSNDASGILPHLGF...,74,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
A8F8N0,"[None, PF09413.15, PF09413.15, PF09413.15, PF0...",MWRILLEEVDLPTANILKSLLEESGIEVLIKPGSFDPVIFGQGGLV...,74,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
Q8A1E5,"[None, None, None, None, None, None, None, PF0...",MKEEDYSKAIEVFSGSPWEAEIIKGLLESNDIRCVIKDGIMGTLAP...,77,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


In [83]:
example_input = torch.tensor(np.stack(subset_df["one_hot"].values[:64]), dtype=torch.float32)

model = ProtENN2_style(num_pfams=len(pfam_to_index))

# example_input.shape

model(example_input).shape

torch.Size([64, 100, 628])

In [84]:
example_output = torch.tensor(np.stack(subset_df["pfam_one_hot"].values[:64]), dtype=torch.float32)

example_output.shape

torch.Size([64, 100, 628])

In [87]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Prepare data
X = torch.tensor(np.stack(subset_df["one_hot"].values[:64]), dtype=torch.float32)  # (batch, 100, 21)
Y_onehot = np.stack(subset_df["pfam_one_hot"].values[:64])  # (batch, 100, num_pfams)
Y = torch.tensor(Y_onehot.argmax(axis=-1), dtype=torch.long)  # (batch, 100)

dataset = TensorDataset(X, Y)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = ProtENN2_style(num_pfams=Y_onehot.shape[-1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()  # expects (N, C) and (N,) or (N, d1, d2, ...) and (N, d1, d2, ...)

# Training loop
for epoch in range(50):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        out = model(xb)  # (batch, 100, num_pfams)
        loss = criterion(out.view(-1, out.shape[-1]), yb.view(-1))  # flatten for CrossEntropyLoss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

Epoch 1, Loss: 3.2313
Epoch 2, Loss: 1.6509
Epoch 3, Loss: 1.3231
Epoch 4, Loss: 1.0535
Epoch 5, Loss: 0.7328
Epoch 6, Loss: 0.6371
Epoch 7, Loss: 0.6618
Epoch 8, Loss: 0.5371
Epoch 9, Loss: 0.5354
Epoch 10, Loss: 0.5361
Epoch 11, Loss: 0.3747
Epoch 12, Loss: 0.2778
Epoch 13, Loss: 0.2132
Epoch 14, Loss: 0.1473
Epoch 15, Loss: 0.1084
Epoch 16, Loss: 0.2242
Epoch 17, Loss: 0.2350
Epoch 18, Loss: 0.3131
Epoch 19, Loss: 0.2638
Epoch 20, Loss: 0.2266
Epoch 21, Loss: 0.1618
Epoch 22, Loss: 0.1212
Epoch 23, Loss: 0.1226
Epoch 24, Loss: 0.0817
Epoch 25, Loss: 0.0775
Epoch 26, Loss: 0.0554
Epoch 27, Loss: 0.0711
Epoch 28, Loss: 0.0304
Epoch 29, Loss: 0.0537
Epoch 30, Loss: 0.0273
Epoch 31, Loss: 0.0142
Epoch 32, Loss: 0.0084
Epoch 33, Loss: 0.0235
Epoch 34, Loss: 0.0180
Epoch 35, Loss: 0.0103
Epoch 36, Loss: 0.0132
Epoch 37, Loss: 0.0083
Epoch 38, Loss: 0.0042
Epoch 39, Loss: 0.0021
Epoch 40, Loss: 0.0021
Epoch 41, Loss: 0.0011
Epoch 42, Loss: 0.0007
Epoch 43, Loss: 0.0009
Epoch 44, Loss: 0.00